In [ ]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

In [ ]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [ ]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [ ]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

In [ ]:
# Load datasets
folds = paths.load_cv_folds(k=5)

In [ ]:
def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Train a ItemKNN with pearson similarity**

In [ ]:
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender

optimizer = ModelOptimizer("ItemKNN_pearson")

STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME + "_pearson"

In [ ]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity":"pearson",
        "topK":optuna_trial.suggest_int("topK", 10, 1500),
        "shrink":optuna_trial.suggest_int("shrink", 0, 2000),
        "normalize":optuna_trial.suggest_categorical("normalize", [True, False]),
        "feature_weighting":optuna_trial.suggest_categorical("feature_weighting", ["BM25", "TF-IDF", "none"])
    }
    
    validation_scores = []
    for URM_train, URM_validation in folds:
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)

        optimizer.log_fold_performance(len(validation_scores), score)
        print(f"  Fold {len(validation_scores)} - Score: {score}")

    return np.mean(validation_scores)

In [ ]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=40
)

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [ ]:
bp = optimizer.get_best_params(STUDY_NAME)

def refined_objective(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity":"pearson",
        "topK":optuna_trial.suggest_int("topK", max(10, bp["topK"]-100), bp["topK"]+100),
        "shrink":optuna_trial.suggest_int("shrink", max(0, bp["shrink"]-200), bp["shrink"]+200),
        "normalize":bp["normalize"],
        "feature_weighting":bp["feature_weighting"]
    }
    
    validation_scores = []
    for URM_train, URM_validation in folds:
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)

        optimizer.log_fold_performance(len(validation_scores), score)
        print(f"  Fold {len(validation_scores)} - Score: {score}")

    return np.mean(validation_scores)

In [ ]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME+"_refined",
    objective_function=refined_objective,
    n_trials=20
)

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

### **Best Model**
- ADD HERE